# Profiling & Quantization — Hands-On Notebook

**Week 11 · CS 203 · Software Tools and Techniques for AI**

Prof. Nipun Batra · IIT Gandhinagar

This notebook covers:
1. Floating point exploration
2. Model parameter counting
3. Profiling (timing, cProfile)
4. Batch size effects
5. Dynamic quantization (PyTorch)
6. ONNX export & benchmarking
7. Final comparison dashboard

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch torchvision onnx onnxruntime numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import struct

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Floating Point Exploration

Let's see how computers actually store decimal numbers.

In [ ]:
# The famous floating point surprise
print(f"0.1 + 0.2 = {0.1 + 0.2}")
print(f"Is 0.1 + 0.2 == 0.3? {0.1 + 0.2 == 0.3}")
print()

# Why? Because 0.1 in binary is infinite: 0.00011001100110011...
# FP32 truncates it. Small errors accumulate.
print(f"0.1 stored as FP32: {np.float32(0.1):.20f}")
print(f"0.1 stored as FP64: {np.float64(0.1):.20f}")

In [ ]:
# See the actual bits of a FP32 number using struct
def show_fp32_bits(value):
    """Show the sign, exponent, and mantissa bits of a FP32 number."""
    packed = struct.pack('>f', value)  # big-endian float
    bits = ''.join(f'{byte:08b}' for byte in packed)
    sign = bits[0]
    exponent = bits[1:9]
    mantissa = bits[9:]
    print(f"Number: {value}")
    print(f"Sign:     {sign} ({'positive' if sign == '0' else 'negative'})")
    print(f"Exponent: {exponent} (= {int(exponent, 2)}, bias=127, actual={int(exponent, 2)-127})")
    print(f"Mantissa: {mantissa}")
    print(f"Full:     {sign} | {exponent} | {mantissa}")
    print()

show_fp32_bits(6.5)
show_fp32_bits(1.0)
show_fp32_bits(-3.14)

In [ ]:
# Compare precision: FP32 vs FP16
values = [0.1, 0.123456789, 3.14159265, 1e-7, 1e7]
print(f"{'Value':<15} {'FP32':>20} {'FP16':>20} {'Difference':>15}")
print("-" * 72)
for v in values:
    fp32 = np.float32(v)
    fp16 = np.float16(v)
    diff = abs(float(fp32) - float(fp16))
    print(f"{v:<15} {float(fp32):>20.10f} {float(fp16):>20.10f} {diff:>15.10f}")

## 2. Model Definition & Parameter Counting

In [ ]:
class SimpleCNN(nn.Module):
    """Small CNN for CIFAR-10-like images (3x32x32)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

model = SimpleCNN()
model.eval()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters:  {total_params:>10,}")
print(f"FP32 size:         {total_params * 4 / 1024:>10.1f} KB")
print(f"FP16 size:         {total_params * 2 / 1024:>10.1f} KB")
print(f"INT8 size:         {total_params * 1 / 1024:>10.1f} KB")

In [ ]:
# Per-layer breakdown
print(f"{'Layer':<35} {'Shape':<25} {'Params':>10} {'Size (KB)':>10}")
print("-" * 85)
for name, param in model.named_parameters():
    n = param.numel()
    print(f"{name:<35} {str(list(param.shape)):<25} {n:>10,} {n * 4 / 1024:>10.1f}")

## 3. Profiling: Timing

In [ ]:
def benchmark(model, input_data, n_runs=200, label="Model"):
    """Benchmark inference time with proper warmup."""
    # Rule 1: Warmup (JIT compilation, cache filling)
    for _ in range(20):
        with torch.no_grad():
            model(input_data)

    # Rule 2: Average many runs
    start = time.perf_counter()
    for _ in range(n_runs):
        with torch.no_grad():
            model(input_data)
    elapsed = (time.perf_counter() - start) / n_runs * 1000

    print(f"{label:<30} {elapsed:>8.2f} ms/batch")
    return elapsed

In [ ]:
# cProfile example: simulating the "slow endpoint" pattern
import cProfile
import io
import pstats

# Simulate loading data + model on every call
def slow_pipeline():
    # Simulating expensive operations
    data = np.random.randn(1000, 64).astype(np.float32)  # simulate read_csv
    x = torch.from_numpy(data[:32].reshape(32, 1, 8, 8).repeat(1, 3, 4, 4))
    with torch.no_grad():
        return model(x)

# Profile it
profiler = cProfile.Profile()
profiler.enable()
for _ in range(10):
    slow_pipeline()
profiler.disable()

# Show results
stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats('cumulative')
stats.print_stats(15)
print(stream.getvalue())

## 4. Batch Size Effects

In [ ]:
# Benchmark at different batch sizes
print("Per-sample latency at different batch sizes:\n")
batch_times = {}
for bs in [1, 4, 16, 32, 64, 128]:
    x = torch.randn(bs, 3, 32, 32)
    ms = benchmark(model, x, label=f"Batch size {bs}")
    batch_times[bs] = ms / bs  # per-sample time

print(f"\nBatching speedup: {batch_times[1] / batch_times[128]:.1f}x faster per sample")

In [ ]:
# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(batch_times.keys()), list(batch_times.values()), 'o-', markersize=8, color='steelblue')
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Time per Sample (ms)', fontsize=12)
ax.set_title('Batching = Free Speedup', fontsize=14, fontweight='bold')
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Dynamic Quantization

In [ ]:
# Save original model
torch.save(model.state_dict(), "model_fp32.pth")
fp32_size = os.path.getsize("model_fp32.pth")

# Dynamic quantization — ONE LINE
quantized_model = torch.quantization.quantize_dynamic(
    model, {nn.Linear}, dtype=torch.qint8
)

torch.save(quantized_model.state_dict(), "model_int8.pth")
int8_size = os.path.getsize("model_int8.pth")

print(f"FP32 model: {fp32_size / 1024:.1f} KB")
print(f"INT8 model: {int8_size / 1024:.1f} KB")
print(f"Compression: {fp32_size / int8_size:.2f}x smaller")

In [ ]:
# Benchmark: FP32 vs INT8
input_data = torch.randn(32, 3, 32, 32)

print("Inference speed comparison:")
fp32_time = benchmark(model, input_data, label="FP32 (original)")
int8_time = benchmark(quantized_model, input_data, label="INT8 (quantized)")
print(f"\nSpeedup: {fp32_time / int8_time:.2f}x")

In [ ]:
# How different are the outputs?
with torch.no_grad():
    fp32_out = model(input_data)
    int8_out = quantized_model(input_data)

agreement = (fp32_out.argmax(1) == int8_out.argmax(1)).float().mean()
max_diff = (fp32_out - int8_out).abs().max().item()

print(f"Prediction agreement: {agreement:.1%}")
print(f"Max output difference: {max_diff:.6f}")
print("\n→ The predictions are nearly identical!")

In [ ]:
# Visualize weight distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

weights = []
for p in model.parameters():
    weights.extend(p.detach().numpy().flatten())

axes[0].hist(weights, bins=100, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].set_title(f'FP32 Weights ({fp32_size/1024:.0f} KB)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Weight Value')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.3)

diffs = (fp32_out - int8_out).detach().numpy().flatten()
axes[1].hist(diffs, bins=50, alpha=0.7, color='coral', edgecolor='white')
axes[1].set_title(f'Quantization Error (max: {max_diff:.4f})', fontsize=13, fontweight='bold')
axes[1].set_xlabel('FP32 output - INT8 output')

plt.tight_layout()
plt.show()

## 6. ONNX Export & Benchmarking

In [ ]:
import onnxruntime as ort

# Export to ONNX
dummy_input = torch.randn(1, 3, 32, 32)
torch.onnx.export(
    model, dummy_input, "model.onnx",
    input_names=["image"], output_names=["logits"],
    dynamic_axes={"image": {0: "batch_size"}},
    opset_version=17,
)
onnx_size = os.path.getsize("model.onnx")
print(f"ONNX model size: {onnx_size / 1024:.1f} KB")

# Load with ONNX Runtime
opts = ort.SessionOptions()
opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
session = ort.InferenceSession("model.onnx", opts)

# Verify it works
test_input = np.random.randn(1, 3, 32, 32).astype(np.float32)
result = session.run(None, {"image": test_input})
print(f"ONNX output shape: {result[0].shape} ✓")

In [ ]:
# Benchmark: PyTorch vs ONNX Runtime
batch_np = np.random.randn(32, 3, 32, 32).astype(np.float32)
batch_torch = torch.from_numpy(batch_np)

# PyTorch
for _ in range(20): 
    with torch.no_grad(): model(batch_torch)
start = time.perf_counter()
for _ in range(200):
    with torch.no_grad(): model(batch_torch)
pytorch_time = (time.perf_counter() - start) / 200 * 1000

# ONNX Runtime
for _ in range(20): session.run(None, {"image": batch_np})
start = time.perf_counter()
for _ in range(200):
    session.run(None, {"image": batch_np})
onnx_time = (time.perf_counter() - start) / 200 * 1000

print(f"PyTorch:      {pytorch_time:.2f} ms/batch")
print(f"ONNX Runtime: {onnx_time:.2f} ms/batch")
print(f"Speedup:      {pytorch_time / onnx_time:.2f}x")

## 7. Final Comparison Dashboard

In [ ]:
# Collect all results
results = {
    "FP32 (PyTorch)": {"size_kb": fp32_size / 1024, "time_ms": fp32_time},
    "INT8 (Quantized)": {"size_kb": int8_size / 1024, "time_ms": int8_time},
    "ONNX Runtime": {"size_kb": onnx_size / 1024, "time_ms": onnx_time},
}

print("\n" + "=" * 65)
print(f"{'Method':<25} {'Size (KB)':>12} {'Latency (ms)':>15} {'Speedup':>10}")
print("=" * 65)
baseline = results["FP32 (PyTorch)"]["time_ms"]
for name, data in results.items():
    speedup = baseline / data["time_ms"]
    print(f"{name:<25} {data['size_kb']:>12.1f} {data['time_ms']:>15.2f} {speedup:>9.2f}x")
print("=" * 65)

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

names = list(results.keys())
sizes = [results[n]["size_kb"] for n in names]
times = [results[n]["time_ms"] for n in names]
colors = ['steelblue', 'coral', 'mediumseagreen']

bars = axes[0].bar(names, sizes, color=colors, edgecolor='white')
axes[0].set_ylabel('Model Size (KB)', fontsize=12)
axes[0].set_title('Model Size Comparison', fontsize=13, fontweight='bold')
for bar, val in zip(bars, sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{val:.0f}', ha='center', fontsize=11)

bars = axes[1].bar(names, times, color=colors, edgecolor='white')
axes[1].set_ylabel('Inference Time (ms/batch)', fontsize=12)
axes[1].set_title('Inference Speed (batch=32)', fontsize=13, fontweight='bold')
for bar, val in zip(bars, times):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}', ha='center', fontsize=11)

plt.tight_layout()
plt.show()
print("\n✅ You now know how to profile, quantize, and export ML models!")